# Generate and embed IDRs

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rotskoff-group/idiom/blob/main/cookbook/notebooks/generate_and_embed.ipynb)

IDiom generates intrinsically disordered regions two ways — **unprompted** (de novo, no context) and
**prompted** (in-filling an IDR between its flanks) — and exposes the residual stream as embeddings
for downstream models. This notebook covers generation, embeddings, and FASTA perplexity scoring.

A GPU is recommended; `DEVICE = "auto"` falls back to CPU.

> **Runtime → Change runtime type → GPU** in Colab before running.

In [1]:
# Install into this notebook's Python environment if IDiom is unavailable.
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("idiom") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "git+https://github.com/rotskoff-group/idiom.git",
    ])

from huggingface_hub import hf_hub_download
import idiom


def example_data(name: str) -> str:
    """Download a cookbook example and return its cached local path."""
    return hf_hub_download(
        "jxliu2/idiom-data", f"example_data/{name}", repo_type="dataset",
    )


print("IDiom loaded from", idiom.__file__)

/data2/scratch/jxliu2/idiom-public/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


IDiom loaded from /data2/scratch/jxliu2/idiom-public/src/idiom/__init__.py


## Parameters

Edit these to point at another model or layer.

In [2]:
from pathlib import Path

MODEL = "jxliu2/idiom-300M"  # Hub ID, released directory, or Lightning checkpoint
DEVICE = "auto"
LAYER = 18  # Zero-based block index; use a smaller index for models with fewer layers.
N = 10
BATCH_SIZE = 4  # Lower this if GPU memory is limited.
SEED = 0
OUT_DIR = Path("generation_outputs")

PERPLEXITY_FASTA = None  # None uses the bundled DisProt example.
PERPLEXITY_MAX_RECORDS = 32  # None scores all valid records.
PERPLEXITY_BATCH_SIZE = 4

In [3]:
from idiom import IDiom

model = IDiom.load(MODEL, device=DEVICE)
if not 0 <= LAYER < model.model.cfg.n_layers:
    raise ValueError(f"LAYER must be between 0 and {model.model.cfg.n_layers - 1}.")
OUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Device: {model.device}; output directory: {OUT_DIR.resolve()}")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 111.33it/s]

Device: cuda; output directory: /data2/scratch/jxliu2/idiom-public/runs/notebook-execution/20260907-143042/generation_outputs


## Unprompted: de novo IDRs

No context at all — the model samples from what it learned an IDR looks like.

In [4]:
idrs = model.generate_unprompted(n=N, temperature=1.0, seed=SEED, batch_size=BATCH_SIZE)

print(f"{len(idrs)} IDRs, mean length {sum(map(len, idrs)) / len(idrs):.0f}")
for i, s in enumerate(idrs, start=1):
    print(f">unprompted_{i} length={len(s)}\n{s}")

10 IDRs, mean length 52
>unprompted_1 length=97
QATPAPQPNLDLSGTSTDMPESYEPSEHVVKKTPKQNRKRKLDPMPEIEKPTNEKKVKVEVKVKVETETQMEEQIIEEPPPKRRKTRKTRKTKSATE
>unprompted_2 length=46
MSPEHETQSYTPTSTPTSSKANRQQGQIAARLKRGETYSLIAGFWK
>unprompted_3 length=52
MGKRLEQQPMYPQYTYYYPHYLQTKQPYAPPPHPMAPPSPSTNSCSQGGAEQ
>unprompted_4 length=31
GAAGAGQPIGGARPAEEPAAAGRPGPAAAEP
>unprompted_5 length=34
MDLPNLISGTTGRLASTSEQPPPFDDRGKNSVNI
>unprompted_6 length=32
ILGKLKSMEFAGSTSSSPNYQFRRKNQQQDQE
>unprompted_7 length=33
REVTGTNRVKFKAGRYSKSPEGSQEFYFFTDPS
>unprompted_8 length=40
KLDFNHRQSLESKEDSMDSGNLKNKATEATKQESADTLAE
>unprompted_9 length=43
MPSHWRQRAAFLTALSAVTLLLVAAPLRAQRKDRAPTPPAAAP
>unprompted_10 length=108
MVFRNPEIWFTVFTALVAHIPNWCTRQMLWSIGSFGASFLFWVISLLKLPNQIGLDGVTIHSLEVQFCSTIIGDSVVVLAVNGTLPHGTASFNNGDVIQNLSSRRLIE


A `length_range` oversamples and filters candidates. Sampling stops at a bounded number of
attempts, so the result can contain fewer than `N` IDRs. This does not constrain the model to
emit a fixed length.

In [5]:
sized = model.generate_unprompted(n=N, length_range=(60, 100), seed=SEED, batch_size=BATCH_SIZE)

print(f"{len(sized)} IDRs, lengths {sorted(len(s) for s in sized)}")
for i, s in enumerate(sized, start=1):
    print(f">length_filtered_{i} length={len(s)}\n{s}")

10 IDRs, lengths [64, 67, 71, 76, 77, 92, 94, 97, 97, 100]
>length_filtered_1 length=97
QATPAPQPNLDLSGTSTDMPESYEPSEHVVKKTPKQNRKRKLDPMPEIEKPTNEKKVKVEVKVKVETETQMEEQIIEEPPPKRRKTRKTRKTKSATE
>length_filtered_2 length=67
MLLNQTIRRGQNQHKNVHLLISMTSKMVLLLITNGYTTSSSLHFSSSSQSNSSSFPLSGTKEESERS
>length_filtered_3 length=71
FDTDRTRPDAQEPPLAAGAFEQAVSTASDIMTEPASQPAPNRKPERDWDDLSEHVTPVDTEKTRANKPTAR
>length_filtered_4 length=92
IINVFEEMPPPELWREVIAGKICIKDHVWKEGIRARLGVPRTFIVLPEEEIPFSIRLMDMISGKKKGVGLSVNFFTPWRAYPNLSIPGEFLR
>length_filtered_5 length=100
RDSFFATESRRHEERGVEVKLTRATFLASVSEVIYPEQADIVHVTSTLDVYFVELSHADVLKTIRDYASSIEFEVFESAPDDDGLLTEKGFVLDGRMERS
>length_filtered_6 length=76
MTNALFSTYNEWWKYNASERERLAQRDGLSQTMTNASKLDRFKKKKRFPKLQDTSTVDKLHQPYHKSTSSCRSKQI
>length_filtered_7 length=94
HVGSCTQLFSKQKLKERAMLPCPWICLARSFPASLNIALVARCRPPSPLPGSAENQPRSGRLVESPLGSGCSVNLYFPGPFCSFYQQAPALSGG
>length_filtered_8 length=77
YGSSSSGSNSTEALKNGKSSENINTDSTGASGAATGIPTTALKPNYNKAAEERLPGVSMETDVLSELESSLNSNDDE
>length_filtered_9 length=

## Prompted: in-fill an IDR between its flanks

Give the model a protein and the coordinates of the disordered span (0-based, half-open) and it
generates replacements for that span, conditioned on the flanking sequence.

In [6]:
protein = "MEDSKVDNRPQACDEFGHIKLMNPQRSTVWYACDEFGHIKLMNPQRST"
idr_start, idr_end = 12, 30

filled = model.generate_prompted(protein, idr_start, idr_end, n=N, seed=SEED, batch_size=BATCH_SIZE)

print(f"left flank  ...{protein[:idr_start][-8:]}")
print(f"right flank {protein[idr_end:][:8]}...")
print(f"\n{len(filled)} in-fills:")
for i, s in enumerate(filled, start=1):
    print(f">infill_{i} length={len(s)}\n{s}")

/data2/scratch/jxliu2/idiom-public/.venv/lib/python3.10/site-packages/torch/utils/_contextlib.py:116: UserWarning: max_new_tokens=1000 exceeds remaining context; limiting to 991 tokens
  return func(*args, **kwargs)


left flank  ...KVDNRPQA
right flank YACDEFGH...

10 in-fills:
>infill_1 length=80
EATPADQPNLDLSGTSTDNGESYEPSEHVVKKTPKQNRKRKLDPMPEIEKETNEKNVEVEVEVKVATETQREEQIIGCPP
>infill_2 length=46
VSPEAETQSYTPTSTPTSEKANPQQGQIAAGLKRGETYSLIAGDNK
>infill_3 length=131
EGKRLSPPAKEYHSPQRAQKSGKPELGRQYCPSSRSPSTNGISYRTEKTDLATTTNGLLDYIPKERGGLSSSSSRSSLHQLSAEDKRTGPYPAPREQRRESTGTNRVKFSAGRYSRSPEGSQEFYPRTDPS
>infill_4 length=31
GAAGAGQPIGGKRGGEEPAAAGRGGAGRAEP
>infill_5 length=36
QDAGIDESPSHDSQGEDGNTGSSAVRGRRRAASRRR
>infill_6 length=36
SAYDDLIFEFPNPEIWFTKRTANGDHIPNWCTRQML
>infill_7 length=38
ADNASTTAAPTPANPPSESHSDAGAEPSHRPVRGASRE
>infill_8 length=540
APELFEPSTAPPFPPTAGYDTDRNGFDYPQISVRPRPQIHPFSGGLARGPFVPIEERRTASRASSLAAMNLESLLRRSRQERMMLGPPRGLDRHIVTPRGTAPSIIPTAKSLNYFPYFPGESDLPKSGNGFPELLTVTQAAEPTTNGGNHYPLPGNSIYQQKGQNMKFVPQNLLTVKPKGKKHPLPQPGRSVFGGSIQIAIRKRALEADRHELNVIRRINRLPRLVSSKIDGDDQTTTLGQSSPLGSSQVSIQGDLLANVINVETPTSAINGQGNFSSSLESPQSSRYHDFEVRTVSVSSGSDDYSVSDYDDDGPQSGGFDDDTRSAVSSSTNEYFEELQADPVSPRASLCAPNQIKSRGKEPKKKSSSSG

### Redesign a real protein

The bundled DisProt records include annotated IDRs and their flanking sequences. Read one record
and generate replacement IDRs. Python spans are **0-based and half-open**; FASTA headers use
**1-based inclusive** spans. The output below contains replacement IDRs only.

In [7]:
from idiom.data.io import read_records

disprot = example_data("disprot/disprot_len1020_idrs.fasta")
record = next(read_records(disprot))

print(f"{record.accession}: {len(record.full_seq)} residues, "
      f"IDR {record.idr_start}-{record.idr_end} ({record.idr_end - record.idr_start} residues)")

redesigned = model.generate_prompted(record.full_seq, record.idr_start, record.idr_end, n=N, seed=SEED, batch_size=BATCH_SIZE)
for i, s in enumerate(redesigned, start=1):
    print(f">{record.accession}_redesigned_{i} length={len(s)}\n{s}")

2026-09-07 14:31:13.473 | INFO     | idiom.data.io:read_fasta:77 - disprot_len1020_idrs.fasta: dropped 2 non-canonical sequence(s)


P03265: 529 residues, IDR 293-334 (41 residues)


/data2/scratch/jxliu2/idiom-public/.venv/lib/python3.10/site-packages/torch/utils/_contextlib.py:116: UserWarning: max_new_tokens=1000 exceeds remaining context; limiting to 533 tokens
  return func(*args, **kwargs)


>P03265_redesigned_1 length=80
LATPADQPNLDLSGTSTDMGESYEPSEHVVKKTPKQNRKRKLDPMPEIEKPTNTKKVKVEVLDKVATPTQREEQIIGCPP
>P03265_redesigned_2 length=46
CSPEAEDQEYTPTSTPFGCKANRQQGQIAARLKRGETYSLIAGFWK
>P03265_redesigned_3 length=34
EGKRLSPPAKKYHAPSRAGASGKPELGRRYCPVS
>P03265_redesigned_4 length=31
GAAGRGQPIGGKRKAEEPVAAGRGGAGRRST
>P03265_redesigned_5 length=36
RRDKRRSGGGGVSATKCLDLPVLIGGTRGRGKSTSE
>P03265_redesigned_6 length=49
RSTAAKSDEDGTWMGELILGKLKDMEFAGSRSSSPNVQFRRKNKQKDKE
>P03265_redesigned_7 length=50
SAKDKRTGPRPAKREKRREETGTNRVKFKAGRYSKSPEGSQEFYPMTDPS
>P03265_redesigned_8 length=42
WEMNPSSSNEPSRNVPSKTDPNHRQSPRRVIKRMTLGQSPSR
>P03265_redesigned_9 length=36
QDAGVDCSPSHWRQRKAFLYASSAVRGRREASPYRF
>P03265_redesigned_10 length=36
SAYPDLIFVFPNPEIWFTKRTANGDHIPNWCTRQML


### Save the generated sequences

Save the de novo IDRs and splice the DisProt replacements into their original flanks. Updated
`_IDR_x-y` headers keep these FASTAs compatible with IDiom's other workflows. Empty generations
are omitted. Files are written to `OUT_DIR` and replaced when this cell is rerun.

For batch generation directly to disk, use `model.generate_unprompted_fasta(...)` or
`model.generate_prompted_fasta(..., return_full=True)`.

In [8]:
idr_path = OUT_DIR / "idrs.fasta"
idr_path.write_text("".join(
    f">generated_{i}_IDR_1-{len(seq)}\n{seq}\n"
    for i, seq in enumerate(idrs) if seq
))
redesigned_path = OUT_DIR / "redesigned.fasta"
redesigned_path.write_text("".join(
    f">{record.accession}_gen{i}_IDR_{record.idr_start + 1}-{record.idr_start + len(seq)}\n"
    f"{record.full_seq[:record.idr_start]}{seq}{record.full_seq[record.idr_end:]}\n"
    for i, seq in enumerate(redesigned) if seq
))
print(f"Saved {idr_path.resolve()} and {redesigned_path.resolve()}")

Saved /data2/scratch/jxliu2/idiom-public/runs/notebook-execution/20260907-143042/generation_outputs/idrs.fasta and /data2/scratch/jxliu2/idiom-public/runs/notebook-execution/20260907-143042/generation_outputs/redesigned.fasta


## Embeddings

`embed` reads the residual stream at whichever layers you ask for. `pool="mean"` gives one vector
per sequence — the usual input to a downstream predictor.

In [9]:
SEQS = [
    "MEDSKVDNRPQACDEFGHIKLMNPQRSTVWY",
    "GSGSQPQPQPGSGSGSNNNNQQQQGSGSGS",
]

pooled, index = model.embed(SEQS, layers=[LAYER], pool="mean")[LAYER]

print(f"pooled: {pooled.shape} -- one {pooled.shape[1]}-d vector per sequence")
print("accessions:", [r["accession"] for r in index])

import json
import numpy as np

np.save(OUT_DIR / "embeddings.npy", pooled)
(OUT_DIR / "embedding_index.json").write_text(json.dumps(index, indent=2))
print("Saved embeddings.npy and embedding_index.json to", OUT_DIR.resolve())

pooled: (2, 1024) -- one 1024-d vector per sequence
accessions: ['seq_0', 'seq_1']
Saved embeddings.npy and embedding_index.json to /data2/scratch/jxliu2/idiom-public/runs/notebook-execution/20260907-143042/generation_outputs


`pool="none"` keeps every residue as its own row. The index carries each row's source position, so
rows stay aligned to residues after any filtering.

In [10]:
per_res, index = model.embed(SEQS[0], layers=[LAYER], pool="none")[LAYER]

print(f"per-residue: {per_res.shape} for a {len(SEQS[0])}-residue sequence")
print("first row:", {k: index[0][k] for k in ("accession", "source_pos", "residue")})

per-residue: (31, 1024) for a 31-residue sequence
first row: {'accession': 'seq_0', 'source_pos': 0, 'residue': 'M'}


## Perplexity

Score a record FASTA under the model's fill-in-the-middle next-token objective. The default uses
up to 32 records from the DisProt example downloaded above; set `PERPLEXITY_FASTA` to evaluate
your own held-out data. This example fixes `prompted_prob=1.0` to score with flanking context.

The result contains mean token NLL in nats, its exponential (`perplexity`), and the number of
scored tokens. Padding is excluded; the objective includes FIM markers and STOP, not just IDR
residues. Compare scores using the same input set and prompting settings.


In [11]:
from idiom.utils.perplexity import perplexity

heldout_fasta = PERPLEXITY_FASTA if PERPLEXITY_FASTA is not None else disprot
scores = perplexity(
    model.model,
    heldout_fasta,
    tokenizer=model.tok,
    device=model.device,
    max_len=model.model.cfg.max_seq_len,
    prompted_prob=1.0,
    max_records=PERPLEXITY_MAX_RECORDS,
    batch_size=PERPLEXITY_BATCH_SIZE,
    num_workers=0,
    seed=SEED,
)
if scores["n_tokens"] == 0:
    raise ValueError("No tokens scored; check the FASTA records and model context length.")
print(f"NLL: {scores['nll']:.3f} nats/token")
print(f"Perplexity: {scores['perplexity']:.3f}")
print(f"Scored tokens: {scores['n_tokens']:,}")

2026-09-07 14:31:15.421 | INFO     | idiom.data.io:read_fasta:77 - disprot_len1020_idrs.fasta: dropped 2 non-canonical sequence(s)


NLL: 1.637 nats/token
Perplexity: 5.139
Scored tokens: 15,064


## Next

- **[`sae_features.ipynb`](https://colab.research.google.com/github/rotskoff-group/idiom/blob/main/cookbook/notebooks/sae_features.ipynb)** — build a feature dataset, rank SAE features, and inspect residue-level activation traces.
- **`cookbook/scripts/`** — pretraining, SFT, and RL templates for real runs.
